# Notebook 06 — RLHF Alignment with PPO

This notebook walks through a simplified RLHF pipeline using TRL (Transformer Reinforcement Learning library). We align a small causal language model using PPO and the reward model from Notebook 05.

In [ ]:
# !pip install trl transformers datasets peft accelerate

## 1. RLHF overview

In [ ]:
# RLHF has three stages:
#   1. Supervised Fine-Tuning (SFT) — train on high-quality demonstrations
#   2. Reward Model Training — learn human preferences (Notebook 05)
#   3. RL Fine-Tuning (PPO) — optimise the SFT model to maximise reward
#      while staying close to the SFT policy (KL penalty)

# PPO objective:
#   L = E[min(r_t * A_t, clip(r_t, 1-eps, 1+eps) * A_t)] - beta * KL(pi || pi_ref)
# where r_t = pi(a|s) / pi_ref(a|s) and A_t is the advantage (reward - baseline)

print("RLHF pipeline: SFT → Reward Model → PPO")

## 2. Load the SFT model and tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2"  # small model for demo
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
print("Model loaded:", MODEL_NAME)

## 3. Prepare the PPO trainer

In [ ]:
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
from datasets import Dataset

ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_NAME)

config = PPOConfig(
    model_name=MODEL_NAME,
    learning_rate=1e-5,
    batch_size=4,
    mini_batch_size=2,
    gradient_accumulation_steps=1,
    ppo_epochs=2,
    log_with=None,
)

# Stub reward function (replace with the reward model from Notebook 05)
def reward_fn(responses: list[str]) -> list[float]:
    return [len(r.split()) / 50.0 for r in responses]  # longer = higher reward (stub)

trainer = PPOTrainer(config=config, model=ppo_model, ref_model=None, tokenizer=tokenizer)
print("PPO trainer ready.")

## 4. Run one PPO step

In [ ]:
import torch

# Sample prompts
prompts = ["Explain neural networks:", "What is fine-tuning?"]
query_tensors = [tokenizer.encode(p, return_tensors="pt").squeeze() for p in prompts]

# Generate responses
response_tensors = [
    trainer.generate(q.unsqueeze(0), max_new_tokens=40, do_sample=True, top_k=50).squeeze()
    for q in query_tensors
]
responses = [tokenizer.decode(r, skip_special_tokens=True) for r in response_tensors]

print("Responses:")
for r in responses:
    print(" -", r[:80])

# Compute rewards
rewards = [torch.tensor(s) for s in reward_fn(responses)]
print("Rewards:", [r.item() for r in rewards])

# PPO update
stats = trainer.step(query_tensors, response_tensors, rewards)
print("PPO step complete. KL:", stats.get("objective/kl", "n/a"))

## 5. Monitor training

In [ ]:
# Key metrics to track during RLHF training:
metrics = {
    "mean_reward": 0.42,          # should increase over training
    "kl_divergence": 0.08,        # should stay < 0.2 (too high = reward hacking)
    "policy_loss": -0.03,         # should decrease
    "value_loss": 0.15,           # should decrease
    "entropy": 2.3,               # should not collapse to 0 (mode collapse)
}

for k, v in metrics.items():
    print(f"{k:20s}: {v}")

# If KL divergence > 0.2, reduce learning rate or increase KL penalty coefficient (beta)